In [ ]:
import math

In [ ]:
class Value:

  def __init__(self,data,_children=(),_op='',label=''):
    self.data = data
    self._prev = set(_children)
    self._op = _op
    self._label = label
    self.grad = 0.0
    self._backward = lambda:None

  def __repr__(self):
    return f'Value(data={self.data})'

  def __add__(self,other):
    other = other if isinstance(other,Value) else Value(other)
    out = Value(self.data + other.data,(self,other),'+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out

  def __radd__(self,other):
    return self + other

  def __mul__(self,other):
    other = other if isinstance(other,Value) else Value(other)
    out = Value(self.data * other.data,(self,other),'*')
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out

  def __rmul__(self,other):
    return self * other

  def __sub__(self,other):
    return self + (-1 * other)

  def __rsub__(self,other):
    return other + (-1 * self)

  def __pow__(self,other):
    assert isinstance(other,(int,float))
    out = Value(self.data ** other,(self,),f'**{other}')
    def _backward():
      self.grad += other * self.data**(other-1) * out.grad
    out._backward = _backward
    return out

  def exp(self):
    out = Value(math.exp(self.data),(self,),'exp')
    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward
    return out

  def __truediv__(self,other):
    other = other if isinstance(other, Value) else Value(other)
    return self * other**-1

  def __rtruediv__(self,other):
    other = other if isinstance(other, Value) else Value(other)
    return other * self**-1

  def tanh(self):
    x = self.data
    t = (math.exp(2*x)-1)/(math.exp(2*x)+1)
    out = Value(t,(self,),'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out

  def backward(self):
    visited = set()
    topo = []
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [ ]:
import random

In [ ]:
class Neuron:

  def __init__(self,nin):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1,1))

  def __call__(self,x):
    act = sum((wi*xi for wi, xi in zip(self.w,x)),self.b)
    out = act.tanh()
    return out

  def parameters(self):
    p = self.w + [self.b]
    return p

class Layer:

  def __init__(self,nin,nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]

  def __call__(self,x):
    outs = [neuron(x) for neuron in self.neurons]
    return outs[0] if len(outs)==1 else outs

  def parameters(self):
    p = [p for neuron in self.neurons for p in neuron.parameters()]
    return p

class MLP:

  def __init__(self,nin,nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i],sz[i+1]) for i in range(len(nouts))]

  def __call__(self,x):
    for layer in self.layers:
      x = layer(x)
    return x

  def parameters(self):
    p = [p for layer in self.layers for p in layer.parameters()]
    return p

n = MLP(3,[4,4,1])

In [ ]:
xs = [[3.0,4.5,2.4],
      [2.0,4.5,5.5],
      [1.0,-4.7,3.8],
      [2.3,4.3,5.2]]
ys = [1.0,1.0,-1.0,1.0]

[Value(data=-0.7239280959892896),
 Value(data=-0.9366355857438167),
 Value(data=-0.7682269567134041),
 Value(data=-0.9364250449831065)]

In [ ]:
for k in range(100):

  yout = [n(x) for x in xs]
  loss = sum((ypred - ygt)**2 for ypred,ygt in zip(yout,ys))

  for p in n.parameters():
    p.grad = 0.0

  loss.backward()

  for p in n.parameters():
    p.data += -0.01 * p.grad

  print(k,loss.data)

0 0.055750022515311186
1 0.05433129773507059
2 0.052981139962062176
3 0.05169474459931654
4 0.05046774316138042
5 0.0492961550852945
6 0.04817634577350102
7 0.047104989950651835
8 0.046079039567321026
9 0.04509569560732239
10 0.04415238325707336
11 0.043246729979464604
12 0.04237654610436099
13 0.04153980760582783
14 0.04073464078460105
15 0.03995930861490169
16 0.03921219854882232
17 0.03849181160030702
18 0.037796752555111966
19 0.03712572117381587
20 0.036477504272556555
21 0.035850968581200016
22 0.0352450542915117
23 0.034658769218940816
24 0.034091183511128234
25 0.033541424844438186
26 0.033008674056898085
27 0.03249216117206634
28 0.031991161773676516
29 0.031504993695544825
30 0.031033013995270246
31 0.030574616183793837
32 0.03012922768597384
33 0.029696307510052217
34 0.02927534410627096
35 0.028865853396998035
36 0.028467376962575595
37 0.0280794803687403
38 0.027701751622913398
39 0.02733379974794294
40 0.026975253463020824
41 0.026625759962510213
42 0.02628498378432237
43

In [ ]:
yout

[Value(data=0.9412416365756585),
 Value(data=0.9362986332561654),
 Value(data=-0.9377522276325961),
 Value(data=0.9390641347203202)]

In [ ]:
n.parameters()

[Value(data=-0.7927677696808529),
 Value(data=0.8575044105943198),
 Value(data=0.1812207237111588),
 Value(data=0.5665979363190278),
 Value(data=0.9205670454083785),
 Value(data=0.7867478529101017),
 Value(data=0.20584425951683727),
 Value(data=0.13255890063236325),
 Value(data=-0.7822712778971554),
 Value(data=-0.8922899387202494),
 Value(data=0.4806831567600337),
 Value(data=0.11109254005565461),
 Value(data=-1.0571970360599763),
 Value(data=-0.38576060120356886),
 Value(data=0.5047793678901927),
 Value(data=-0.1802694469730393),
 Value(data=0.1637820840380681),
 Value(data=0.36992360944954894),
 Value(data=-0.41858518442957),
 Value(data=-0.8441669976253213),
 Value(data=-0.934535903099416),
 Value(data=0.803567610760338),
 Value(data=-0.8585845255474495),
 Value(data=0.46251086172398204),
 Value(data=0.761966146426087),
 Value(data=-0.9373389439922649),
 Value(data=0.2135634029360352),
 Value(data=0.003416376216571768),
 Value(data=-0.5321317471244039),
 Value(data=-0.9307169485055